# Step 6 — Version B on GPU: vLLM on a Kaggle T4

Part of **llm-serving-at-scale**. This notebook:
1. starts **vLLM** (continuous batching + PagedAttention) on **one** T4 GPU,
2. runs a **batch-size sweep** (1 → 256 concurrent requests) and compares it with llama.cpp on CPU,
3. runs the **same Locust load test** used for every other version (10 / 100 / 1000 users),
4. zips the results so you can copy them back into the repo.

**Before running — notebook settings (right sidebar):**
- **Accelerator: `GPU T4 x2`** (not P100: it's compute capability 6.0, below vLLM's 7.5 minimum). We use one of the two GPUs.
- **Internet: On** (needs a phone-verified Kaggle account).

Then **Run All**. Expect ~35–45 min of GPU time (install ~5, sweep ~3, load tests ~25).

In [ ]:
# ---- configuration -------------------------------------------------------------
REPO_URL = "https://github.com/rohithram22/llm-serving-at-scale.git"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
VLLM_VERSION = "0.18.1"      # known-good on Kaggle T4 (Sep 2026). If the install fails, set "" for latest.
GPU_MEM_UTIL = 0.85          # fraction of GPU memory vLLM may use (weights + KV cache + activations)
MAX_MODEL_LEN = 4096         # max tokens per request (prompt + answer)
MAX_NUM_SEQS = 256           # max requests decoded together = the continuous-batching batch limit
SWEEP_LEVELS = [1, 2, 4, 8, 16, 32, 64, 128, 256]
USERS_LIST = [10, 100, 1000]
LOAD_DURATION_S = 180
LABEL = "vllm_t4_fp16"
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/llm-serving-at-scale"
PORT = 8000

## 1. Check the GPU

In [ ]:
import subprocess

def sh(cmd: str, check: bool = True) -> str:
    """Run a shell command, print and return its output."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (p.stdout + p.stderr).strip()
    if out:
        print(out)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return out

gpus = sh("nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv,noheader")
cap = float(gpus.splitlines()[0].split(",")[-1])
assert cap >= 7.5, (f"GPU compute capability {cap} < 7.5: vLLM won't run. "
                    "Switch the accelerator to 'GPU T4 x2' in the notebook settings.")
print(f"\nOK: compute capability {cap}. We'll use GPU 0 only (one replica).")

## 2. Install vLLM + Locust, clone the repo (for the shared load-test harness)

In [ ]:
import os, sys
spec = f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"
sh(f"{sys.executable} -m pip install -q {spec} locust 2>&1 | tail -3", check=False)
sh(f"{sys.executable} -m pip show vllm | head -2")
if not os.path.isdir(REPO_DIR):
    sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
os.makedirs("results/raw", exist_ok=True)
print("working in", os.getcwd())

## 3. Start vLLM

Flags worth knowing:
- `--dtype half`: T4 has no bf16 support, so fp16.
- `--gpu-memory-utilization`: vLLM grabs this share of GPU memory up front, loads the weights, and turns **everything left into KV-cache blocks** (PagedAttention).
- `--max-num-seqs`: the most requests it will decode in one step.

The startup log tells you the KV-cache size and how many full-length requests fit at once; it's printed below.

In [ ]:
import time, urllib.request

LOG = f"{WORK}/vllm.log"

def stop_vllm():
    subprocess.run("pkill -f 'vllm serve' ; sleep 3", shell=True)

def wait_health(timeout_s: int = 900) -> float:
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            if urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2).status == 200:
                return time.time() - t0
        except Exception:
            pass
        time.sleep(3)
    print(sh(f"tail -40 {LOG}", check=False))
    raise TimeoutError("vLLM did not become healthy; see the log above")

def start_vllm():
    stop_vllm()
    cmd = (f"CUDA_VISIBLE_DEVICES=0 nohup vllm serve {MODEL} --dtype half --port {PORT} "
           f"--max-model-len {MAX_MODEL_LEN} --gpu-memory-utilization {GPU_MEM_UTIL} "
           f"--max-num-seqs {MAX_NUM_SEQS} > {LOG} 2>&1 &")
    subprocess.run(cmd, shell=True)
    secs = wait_health()
    print(f"vLLM ready after {secs:.0f}s")
    return secs

startup_s = start_vllm()
print("\n--- key startup facts from the log ---")
_ = sh(f"grep -iE 'KV cache|maximum concurrency|attention backend|cuda graph|dtype' {LOG} | tail -12", check=False)

## 4. Smoke test: same prompt as Version A / llama.cpp

In [ ]:
import json
req = {"model": MODEL, "messages": [{"role": "user", "content": "What is the capital of Japan?"}],
       "max_tokens": 32, "temperature": 0}
t0 = time.time()
r = urllib.request.urlopen(urllib.request.Request(
    f"http://localhost:{PORT}/v1/chat/completions", data=json.dumps(req).encode(),
    headers={"Content-Type": "application/json"}), timeout=60)
body = json.loads(r.read())
print(f"{time.time() - t0:.2f}s ->", body["choices"][0]["message"]["content"])
print("usage:", body["usage"])
print("prompt_tokens should be 36 (same chat template as Versions A and B-cpu):",
      body["usage"]["prompt_tokens"])

## 5. Batch-size sweep: does batching keep scaling on a GPU?

Same prompt and **exactly 32 output tokens** (`ignore_eos`) as the CPU probes in Steps 3 and 5.
For each level we fire *n* requests at once and measure total tokens/s and per-request latency.

In [ ]:
import asyncio
import httpx
import pandas as pd

PROMPT = "Explain what an API is to a beginner."

async def _one(client):
    t0 = time.perf_counter()
    r = await client.post(f"http://localhost:{PORT}/v1/chat/completions", json={
        "model": MODEL, "messages": [{"role": "user", "content": PROMPT}],
        "max_tokens": 32, "temperature": 0, "ignore_eos": True})
    r.raise_for_status()
    return time.perf_counter() - t0, r.json()["usage"]["completion_tokens"]

async def sweep(levels):
    rows = []
    limits = httpx.Limits(max_connections=1024, max_keepalive_connections=1024)
    async with httpx.AsyncClient(timeout=300, limits=limits) as client:
        await asyncio.gather(*[_one(client) for _ in range(8)])          # warm-up
        for n in levels:
            t0 = time.perf_counter()
            res = await asyncio.gather(*[_one(client) for _ in range(n)])
            wall = time.perf_counter() - t0
            lat = [x[0] for x in res]
            toks = sum(x[1] for x in res)
            rows.append({"concurrency": n, "wall_s": round(wall, 3),
                         "req_per_s": round(n / wall, 2), "tokens_per_s": round(toks / wall, 1),
                         "mean_latency_s": round(sum(lat) / n, 3),
                         "ms_per_step": round(1000 * (sum(lat) / n) / 32, 1)})
            print(rows[-1])
    return pd.DataFrame(rows)

sweep_df = await sweep(SWEEP_LEVELS)
base = sweep_df.loc[0, "tokens_per_s"]
sweep_df["speedup_vs_1"] = (sweep_df["tokens_per_s"] / base).round(1)
sweep_df.to_csv(f"results/{LABEL}_batch_sweep.csv", index=False)
sweep_df

## 6. Chart: GPU (vLLM) vs CPU (llama.cpp, Step 5)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

# CPU numbers measured in Step 5 (results/llamacpp_fp16_probe.txt): 4 vCPU, 2 threads, 32 tokens.
cpu = pd.DataFrame({"concurrency": [1, 4, 8], "tokens_per_s": [27.8, 50.9, 64.1],
                    "mean_latency_s": [1.12, 2.48, 3.96]})
gpu = sweep_df

BLUE, ORANGE = "#2a78d6", "#eb6834"                      # validated categorical slots 1-2
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e7e6e2", "#fcfcfb"

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), facecolor=SURFACE)
panels = [("tokens_per_s", "Throughput (output tokens/s, log scale)", True),
          ("mean_latency_s", "Latency per request (s)", False)]
for ax, (col, title, logy) in zip(axes, panels):
    ax.set_facecolor(SURFACE)
    for df, color, name in [(gpu, BLUE, "GPU · vLLM · T4"), (cpu, ORANGE, "CPU · llama.cpp · 2 cores")]:
        ax.plot(df["concurrency"], df[col], color=color, lw=2, marker="o", ms=8,
                markeredgecolor=SURFACE, markeredgewidth=2, label=name, zorder=3)
        last = df.iloc[-1]
        ax.annotate(f"{last[col]:,.0f}" if col == "tokens_per_s" else f"{last[col]:.2f}s",
                    (last["concurrency"], last[col]), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9, color=INK2)
    ax.set_xscale("log", base=2)
    ticks = sorted(set(gpu["concurrency"]) | set(cpu["concurrency"]))
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(v)}"))
    ax.xaxis.set_minor_locator(NullLocator())
    if logy:
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
        ax.yaxis.set_minor_formatter(FuncFormatter(lambda v, _: ""))
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.set_xlabel("concurrent requests (= batch size)", color=INK2, fontsize=9)
    ax.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.tick_params(colors=INK2, labelsize=9)
    for s in ax.spines.values():
        s.set_visible(False)
axes[0].legend(frameon=False, fontsize=9, labelcolor=INK)
fig.suptitle("Batch size vs throughput and latency: GPU (vLLM, T4) vs CPU (llama.cpp)",
             x=0.01, ha="left", fontsize=13, color=INK)
fig.tight_layout()
fig.savefig(f"results/batch_scaling_gpu_vs_cpu.png", dpi=150, facecolor=SURFACE)
plt.show()

## 7. Load test — same harness, same prompts, same settings as every other version

A fresh vLLM server for each user count, so one run's backlog can't pollute the next.
After each run we print vLLM's own counters (queue length, prefix-cache hits).

In [ ]:
import re

def vllm_counters(path):
    keep = ("num_requests_running", "num_requests_waiting", "prefix_cache_queries",
            "prefix_cache_hits", "kv_cache_usage", "gpu_cache_usage", "num_preemptions")
    vals = {}
    for line in open(path):
        if line.startswith("#"):
            continue
        m = re.match(r"vllm:([a-z_]+)(?:\{[^}]*\})?\s+([0-9.eE+-]+)$", line.strip())
        if m and any(k in m.group(1) for k in keep):
            vals[m.group(1)] = vals.get(m.group(1), 0.0) + float(m.group(2))
    return vals

for users in USERS_LIST:
    start_vllm()
    sh(f"bash loadtest/run_load.sh {LABEL} {users} {LOAD_DURATION_S} http://localhost:{PORT}", check=False)
    mfile = f"results/raw/{LABEL}_u{users}_metrics.txt"
    if os.path.exists(mfile) and os.path.getsize(mfile):
        c = vllm_counters(mfile)
        hits, queries = c.get("prefix_cache_hits_total", 0), c.get("prefix_cache_queries_total", 0)
        print(f"vLLM counters @ {users} users: waiting={c.get('num_requests_waiting')} "
              f"running={c.get('num_requests_running')} preemptions={c.get('num_preemptions_total')} "
              f"prefix-cache hit rate={hits / queries:.0%}" if queries else f"vLLM counters: {c}")
    sh(f"cp {LOG} results/raw/{LABEL}_u{users}_server.log", check=False)
stop_vllm()

## 8. Compare with every version so far

In [ ]:
_ = sh("python loadtest/show_summary.py version_a_guarded llamacpp vllm", check=False)

## 9. Save the results → download `step6_results.zip` from the Output panel

In [ ]:
import glob, zipfile
env = sh(f"nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader; "
         f"{sys.executable} -m pip show vllm torch 2>/dev/null | grep -E '^(Name|Version)'; nproc; free -h | head -2",
         check=False)
with open(f"results/{LABEL}_environment.txt", "w") as f:
    f.write(env + f"\nvLLM startup to healthy: {startup_s:.0f}s\n")
sh(f"grep -iE 'KV cache|maximum concurrency|attention backend' {LOG} > results/{LABEL}_startup_facts.txt", check=False)

files = (["results/loadtest_summary.csv", "results/batch_scaling_gpu_vs_cpu.png"]
         + glob.glob(f"results/{LABEL}_*"))
zpath = f"{WORK}/step6_results.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for p in files:
        if os.path.exists(p):
            z.write(p)
print(f"saved {zpath} with:", *files, sep="\n  ")